# Classifier evaluation

This notebook renders report figures from exported evaluation metrics only.

In [ ]:
from pathlib import Path
import json
import os
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from notebooks.utils.classification_experiment_utils import discover_model_artifacts, evaluate_feature_vector_model
from notebooks.utils.complete_classifier_analysis_utils import (detector_confusion_matrices, select_report_metrics, targeted_evaluation_status)
from notebooks.utils.feature_aggregation_utils import load_prepared_partition, prepared_partition_path
from notebooks.utils.model_training_utils import load_model_artifact, load_model_metadata
from notebooks.utils.report_diagrams_utils import (available_metric_rows, report_output_paths, save_report_figure, save_report_table)

DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
REPORT_RUN_ID = os.environ.get("REPORT_RUN_ID", "report-regeneration-2026-10-26")
OUTPUTS = report_output_paths(DATA_ROOT, report_run_id=REPORT_RUN_ID)
RUN_ID = "feature-vector-classification-2026-09-25-logreg-random-forest-max600-min100"
RESULT_ROOT = DATA_ROOT / "experiment-results/classification/evaluations" / RUN_ID
REPORT_CONFIGURATION = {
    "scorer_llm": "gemma_3_270m_it",
    "feature_set": "simple_similarity",
    "training_regime": "Kaggle+MAGE+Outfox",
    "training_window": 32,
    "partition": "test",
}
ANALYSIS_ROOT = DATA_ROOT / "experiment-results/classification/analysis/complete-analysis-feature-vector-classification-2026-09-25-logreg-random-forest-max600-min100/output"
sns.set_theme(style="whitegrid", context="paper")

In [ ]:
all_metrics = pd.read_csv(RESULT_ROOT / "evaluation_metrics.csv")
metrics = select_report_metrics(all_metrics, **REPORT_CONFIGURATION)
metrics = available_metric_rows(metrics, "macro_f1")
save_report_table(metrics, OUTPUTS.tables / "classifier_evaluation_metrics.csv")
for name in ("binary_metrics", "robustness_metrics", "operating_points"):
    path = ANALYSIS_ROOT / f"{name}.csv"
    if path.is_file():
        save_report_table(pd.read_csv(path), OUTPUTS.tables / f"{name}.csv")

BBC_DATASET = "bbc_news_content"
BBC_ARTIFACT_RUN_ID = "feature-vector-classification-2026-09-25-logreg-random-forest-max600-min100"
MODELS_ROOT = DATA_ROOT / "experiment-results/classification/models"
bbc_rows = []
for model_family in ("logistic_regression", "random_forest"):
    candidates = discover_model_artifacts(MODELS_ROOT, filters={
        "model_family": model_family, "scorer_llm": REPORT_CONFIGURATION["scorer_llm"],
        "feature_set": REPORT_CONFIGURATION["feature_set"], "training_regime": REPORT_CONFIGURATION["training_regime"],
        "training_window": str(REPORT_CONFIGURATION["training_window"]), "run_id": BBC_ARTIFACT_RUN_ID,
    })
    if not candidates:
        bbc_rows.append(targeted_evaluation_status(pd.DataFrame(columns=all_metrics.columns), dataset=BBC_DATASET, detector_families=[model_family], **REPORT_CONFIGURATION))
        continue
    artifact_path = candidates[0]
    artifact_metadata = load_model_metadata(artifact_path, expected={"model_family": model_family, "scorer_llm": REPORT_CONFIGURATION["scorer_llm"], "feature_set": REPORT_CONFIGURATION["feature_set"], "training_regime": REPORT_CONFIGURATION["training_regime"], "training_window": str(REPORT_CONFIGURATION["training_window"]), "run_id": BBC_ARTIFACT_RUN_ID})
    frames = []
    for split in ("HG",):
        path = prepared_partition_path(DATA_ROOT, scorer_llm=REPORT_CONFIGURATION["scorer_llm"], dataset=BBC_DATASET, split=split, feature_set=REPORT_CONFIGURATION["feature_set"], window_configuration=REPORT_CONFIGURATION["training_window"])
        prepared_metadata = json.loads((path / "metadata.json").read_text(encoding="utf-8"))
        frames.append(load_prepared_partition(path, expected=prepared_metadata))
    bbc_metrics, _, _ = evaluate_feature_vector_model(load_model_artifact(artifact_path, expected=artifact_metadata, trust_artifact=True), {BBC_DATASET: pd.concat(frames, ignore_index=True)}, partition=REPORT_CONFIGURATION["partition"], metadata=artifact_metadata)
    bbc_metrics = bbc_metrics.assign(evaluation_regime="BBC", evaluation_window=str(REPORT_CONFIGURATION["training_window"]), artifact_training_regime=REPORT_CONFIGURATION["training_regime"], training_regime=REPORT_CONFIGURATION["training_regime"])
    bbc_rows.append(bbc_metrics)
bbc_evaluation = pd.concat(bbc_rows, ignore_index=True) if bbc_rows else pd.DataFrame()
save_report_table(bbc_evaluation, OUTPUTS.tables / "bbc_gemma270m_evaluation.csv")

In [ ]:
summary = metrics.groupby(["model_family", "scorer_llm", "feature_set", "training_regime", "training_window", "partition", "dataset", "evaluation_window"], as_index=False)[["roc_auc", "pr_auc", "macro_f1", "balanced_accuracy"]].mean()
save_report_table(summary, OUTPUTS.tables / "classifier_metric_summary.csv")
long = summary.melt(id_vars=["model_family", "dataset", "evaluation_window"], value_vars=["roc_auc", "pr_auc", "macro_f1", "balanced_accuracy"], var_name="metric", value_name="score")
long["dataset"] = long["dataset"].astype(str)
long["metric"] = long["metric"].astype(str)
figure, axis = plt.subplots(figsize=(12, 6), constrained_layout=True)
sns.barplot(data=long, x="dataset", y="score", hue="metric", errorbar=None, ax=axis)
axis.set(title="Classifier performance on held-out test rows", ylim=(0, 1))
save_report_figure(figure, OUTPUTS.figures / "classifier_metric_comparison")
plt.close(figure)

In [ ]:
MIXED_MATRIX_CONFIGURATION = {"evaluation_regime": "Kaggle+MAGE+Outfox", "evaluation_window": 32, "partition": "test"}
MIXED_BBC_MATRIX_CONFIGURATION = {"evaluation_regime": "Kaggle+MAGE+Outfox+BBC", "evaluation_window": 32, "partition": "test"}
CONFUSION_OUTPUT_NAMES = {"logistic_regression": "confusion_matrix_logistic_regression", "random_forest": "confusion_matrix_random_forest"}
CONFUSION_OUTPUT_NAMES_MIXED_BBC = {"logistic_regression": "confusion_matrix_logistic_regression_mixed_bbc", "random_forest": "confusion_matrix_random_forest_mixed_bbc"}
matrix_metrics = {}
for matrix_name, matrix_configuration in (("mixed", MIXED_MATRIX_CONFIGURATION), ("mixed_bbc", MIXED_BBC_MATRIX_CONFIGURATION)):
    matrix_metrics[matrix_name] = metrics.loc[
        metrics["evaluation_regime"].eq(matrix_configuration["evaluation_regime"])
        & metrics["evaluation_window"].astype("string").eq(str(matrix_configuration["evaluation_window"]))
        & metrics["partition"].eq(matrix_configuration["partition"])
    ].copy()
for matrix_name, matrix_frame in matrix_metrics.items():
    for model_family, confusion in detector_confusion_matrices(matrix_frame).items():
        output_names = CONFUSION_OUTPUT_NAMES if matrix_name == "mixed" else CONFUSION_OUTPUT_NAMES_MIXED_BBC
        output_name = output_names[model_family]
        save_report_table(confusion.rename_axis("true_class").reset_index(), OUTPUTS.tables / f"{output_name}.csv")
        figure, axis = plt.subplots(figsize=(5, 4), constrained_layout=True)
        sns.heatmap(confusion, annot=True, fmt=".3f", vmin=0, vmax=1, cmap="Blues", ax=axis)
        axis.set(title=f"Normalized {model_family} confusion matrix ({matrix_name})", xlabel="Predicted class", ylabel="True class")
        save_report_figure(figure, OUTPUTS.figures / output_name)
        plt.close(figure)